# LLaDA-o Multimodal Demo

This notebook provides a unified, reproducible inference workflow covering the following three capabilities:

- Text-to-image
- Image understanding
- Image editing

The reference image generated in Step 1 is used directly as the input for Steps 2 and 3. All outputs are saved to `demo_outputs/` by default.


In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import display

CANDIDATES = [Path.cwd(), Path.cwd() / "LLaDA-o"]
PROJECT_DIR = next((path for path in CANDIDATES if (path / "inferencer.py").exists()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Please launch the notebook from the repo root or the LLaDA-o directory.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from demo_pipeline import (
    DEFAULT_IMAGE_EDIT_ARGS,
    DEFAULT_TEXT_TO_IMAGE_ARGS,
    DEFAULT_UNDERSTANDING_ARGS,
    LLaDAMultimodalDemo,
    load_image,
    set_seed,
)

def save_text(path, text):
    path.write_text(text.strip() + "\n", encoding="utf-8")

print(f"Project dir: {PROJECT_DIR}")


In [ ]:
# First download the model from https://huggingface.co/GSAI-ML/LLaDA-o
# Then set MODEL_PATH to the local directory you downloaded, or set LLADAO_MODEL_PATH.
MODEL_PATH = os.environ.get("LLADAO_MODEL_PATH", "/path/to/local/GSAI-ML-LLaDA-o")
MAX_MEM_PER_GPU = "40GiB"
OFFLOAD_DIR = "/tmp/lladao_offload"
OUTPUT_DIR = PROJECT_DIR / "demo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

TEXT_TO_IMAGE_PROMPT = "A studio-quality product photo of a glass teapot shaped like a tiny planet, soft lighting."
UNDERSTANDING_PROMPT = "Describe this image in detail."
IMAGE_EDIT_PROMPT = (
    "Turn the Earth-themed glass teapot into a Mars-themed glass teapot, "
    "keep the transparent glass material, studio product photography, and soft lighting."
)

TEXT_TO_IMAGE_ARGS = dict(DEFAULT_TEXT_TO_IMAGE_ARGS)
UNDERSTANDING_ARGS = dict(DEFAULT_UNDERSTANDING_ARGS)
IMAGE_EDIT_ARGS = dict(DEFAULT_IMAGE_EDIT_ARGS)
BATCH_PROMPT_FILE = PROJECT_DIR / "prompt.txt"
BATCH_OUTPUT_DIR = OUTPUT_DIR / "batch_text_to_image"
BATCH_START_LINE = 0

print(f"Model path: {MODEL_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Batch prompt file: {BATCH_PROMPT_FILE}")
print(f"Batch output dir: {BATCH_OUTPUT_DIR}")
if MODEL_PATH == "/path/to/local/GSAI-ML-LLaDA-o":
    print("First download the model from https://huggingface.co/GSAI-ML/LLaDA-o, then change MODEL_PATH to the downloaded local path, or set LLADAO_MODEL_PATH in your environment.")


## Load Model

Load one shared model instance so the three tasks can reuse the same initialization.


In [ ]:
demo = LLaDAMultimodalDemo.from_pretrained(
    model_path=MODEL_PATH,
    max_mem_per_gpu=MAX_MEM_PER_GPU,
    offload_dir=OFFLOAD_DIR,
)

print("Model ready.")


## 1. Text-to-Image

Generate a reference image first. The later image-understanding and image-editing steps both use this image as input.


In [ ]:
text_to_image_result = demo.text_to_image(
    TEXT_TO_IMAGE_PROMPT,
    seed=SEED,
    **TEXT_TO_IMAGE_ARGS,
)

reference_image = text_to_image_result["image"]
text_to_image_path = OUTPUT_DIR / "01_text_to_image.png"
reference_image.save(text_to_image_path)

display(reference_image)
print(f"Saved: {text_to_image_path}")


## 2. Image Understanding

Use the reference image from the previous section as input, generate a detailed textual description, and save it to a text file.


In [ ]:
understanding_result = demo.understand(
    reference_image,
    UNDERSTANDING_PROMPT,
    **UNDERSTANDING_ARGS,
)

understanding_path = OUTPUT_DIR / "02_understanding.txt"
save_text(understanding_path, understanding_result["text"])

display(reference_image)
print(understanding_result["text"])
print(
    f"valid/total tokens: {understanding_result['valid_tokens']}/{understanding_result['total_tokens']} | "
    f"elapsed: {understanding_result['elapsed_seconds']:.2f}s"
)
print(f"Saved: {understanding_path}")


## 3. Image Editing

Continue using the same reference image as input and perform one themed edit so you can compare the generated and edited results.


In [ ]:
image_edit_result = demo.edit_image(
    reference_image,
    IMAGE_EDIT_PROMPT,
    seed=SEED,
    **IMAGE_EDIT_ARGS,
)

edited_image = image_edit_result["image"]
image_edit_path = OUTPUT_DIR / "03_image_edit.png"
edited_image.save(image_edit_path)

print("Source image:")
display(reference_image)
print("Edited image:")
display(edited_image)
print(f"Saved: {image_edit_path}")


## Notes

- The configuration section keeps default prompts for text-to-image, image understanding, and image editing, and you can modify them directly as needed.
- Modify `TEXT_TO_IMAGE_ARGS`, `UNDERSTANDING_ARGS`, and `IMAGE_EDIT_ARGS` to run parameter experiments.
- Batch generation reads prompts from `BATCH_PROMPT_FILE`, and the outputs are saved to `BATCH_OUTPUT_DIR`.
- If you want to use your own image for understanding or editing, replace `reference_image` with `load_image("/absolute/path/to/image.png")`.


## 4. Batch Text-to-Image from `prompt.txt`

This section reads prompts from `prompt.txt` line by line and writes the outputs to a subdirectory under `demo_outputs/`.


In [ ]:
if not BATCH_PROMPT_FILE.exists():
    raise FileNotFoundError(f"Prompt file not found: {BATCH_PROMPT_FILE}")

with open(BATCH_PROMPT_FILE, "r", encoding="utf-8") as f:
    batch_prompts = [line.strip() for line in f if line.strip()]

if not batch_prompts:
    raise ValueError(f"No valid prompts found in: {BATCH_PROMPT_FILE}")

BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)

saved_paths = []
last_image = None
total_prompts = len(batch_prompts)

for i, prompt in enumerate(batch_prompts):
    if i < BATCH_START_LINE:
        continue

    result = demo.text_to_image(prompt, **TEXT_TO_IMAGE_ARGS)
    save_path = BATCH_OUTPUT_DIR / f"{i + 1:03d}.png"
    result["image"].save(save_path)

    saved_paths.append(save_path)
    last_image = result["image"]
    print(f"[{i + 1}/{total_prompts}] Saved: {save_path}")

print(f"Generated {len(saved_paths)} images into: {BATCH_OUTPUT_DIR}")
if last_image is not None:
    display(last_image)
